# 小时级交通预测 · CatBoost 分位数回归

本 notebook 实现 [`model.md`](model.md) 方案：用 `data_autobahn/` 数据预测 **2026–2029 每天每小时**、**12 个站点** 的：

- `kfz_h` 总流量（P10/P50/P90 分位回归）
- `sv_h` 大车流量（占比法）
- `v_kfz` 平均车速（自由流基准 − 降速 两段式）

**核心思想**：把 2023–2025 主流量数据凝练成「历史画像特征」，CatBoost 在其之上用 conditional 数据（假期/天气/施工/事件）做偏移修正。无 lag、无递归。

**工程特性**：tqdm 进度条 + ETA、CatBoost 快照续训、清晰的 train/val loss 曲线、时序 hold-out 评估。

---

### 运行顺序
1. 库导入
2. **超参数配置**（§1.1 — 全部在此修改）
3. 路径与数据文件
4. 数据加载与清洗
5. 特征工程（日历 + 历史画像 + conditional）
6. CatBoost 训练（kfz_h 分位）
7. Loss 曲线
8. sv_h / v_kfz 模型
9. 评估与保存

## 1. 库导入

> 若缺少依赖，先运行：`pip install catboost pandas numpy matplotlib tqdm pyarrow`

**超参数全部在下一节单独配置块中修改。**

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from tqdm.auto import tqdm

from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

## 1.1 超参数配置（★ 全部在此修改）

训练、特征、评估、可视化的可调参数集中于此，下方代码块之外不应硬编码超参。

In [ ]:
# =============================================================================
# 超参数配置块 — 修改此处即可，无需在 notebook 其他位置找 magic number
# =============================================================================

# --- 随机种子 ---
RANDOM_SEED = 42

# --- 时序切分（禁止随机划分）---
TRAIN_END = pd.Timestamp("2024-12-31 23:59:59")   # 训练: 2023-01-01 ~ 2024-12-31
VAL_START = pd.Timestamp("2025-01-01 00:00:00")    # 验证: 2025 全年

# --- CatBoost 通用超参（kfz / lkw / speed 共享底座）---
CB_PARAMS = dict(
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    min_data_in_leaf=100,
    subsample=0.8,
    random_seed=RANDOM_SEED,
    early_stopping_rounds=150,
    task_type="CPU",
    allow_writing_files=True,
)

# --- 各目标模型覆盖（在 CB_PARAMS 之上 merge）---
CB_KFZ_PARAMS = {}   # kfz_h 分位回归，loss 在训练循环里按 quantile 注入
CB_LKW_PARAMS = dict(loss_function="RMSE", eval_metric="RMSE")   # lkw_ratio
CB_SPD_PARAMS = dict(loss_function="RMSE", eval_metric="RMSE")   # speed_drop

# --- kfz_h 分位数 ---
QUANTILES = {"p10": 0.1, "p50": 0.5, "p90": 0.9}

# --- 训练工程（快照续训 / 进度条 / 回退）---
USE_SNAPSHOT = True
SNAPSHOT_INTERVAL_SEC = 30          # 快照保存间隔（秒）
FALLBACK_VERBOSE = 200              # callback 失败时 CatBoost 原生 verbose 间隔

# --- 历史画像聚合分位 ---
PROFILE_KFZ_P90 = 0.90              # prof_kfz_p90
PROFILE_V_P85 = 0.85                # prof_v_p85 自由流基准

# --- 预测后处理截断 ---
KFZ_CLIP_MIN = 0.0                  # 流量下限
LKW_RATIO_CLIP_MIN = 0.0            # 重型车占比下限
SPEED_CLIP_MIN = 20.0               # 车速下限 (km/h)
SPEED_CLIP_MAX = 160.0              # 车速上限 (km/h)

# --- 评估指标 ---
MAPE_EPS = 1.0                      # MAPE 分母保护（y > eps）
PEAK_QUANTILE = 0.90                # 峰值小时 top 10%
PICP_TARGET_PCT = 80.0              # P10–P90 目标覆盖率 (%)

# --- 可视化 ---
PLOT_WEEK_START = "2025-08-01"
PLOT_WEEK_END = "2025-08-08"
PLOT_FIGSIZE = (13, 4.5)
FI_TOP_K = 20                       # 特征重要性展示条数
LOSS_FIGSIZE_PER_PANEL = (5.2, 4.2)

# --- 推理演示 ---
DEMO_FORECAST_DATE = "2026-08-01"

# =============================================================================
np.random.seed(RANDOM_SEED)

print("超参数已加载 ✔")
print(f"  训练截止: {TRAIN_END.date()}  |  验证起始: {VAL_START.date()}")
print(f"  CatBoost: iterations={CB_PARAMS['iterations']}, lr={CB_PARAMS['learning_rate']}, depth={CB_PARAMS['depth']}")
print(f"  分位数: {QUANTILES}")

## 1.2 路径与数据文件

In [ ]:
def find_root() -> Path:
    here = Path.cwd()
    for cand in [here, *here.parents]:
        if (cand / "data_autobahn").exists():
            return cand
    raise FileNotFoundError("未找到 data_autobahn 目录")


ROOT = find_root()
DATA_DIR = ROOT / "data_autobahn"
MODEL_DIR = ROOT / "models"
SNAP_DIR = ROOT / "models" / "snapshots"
PROC_DIR = ROOT / "processed"
for d in (MODEL_DIR, SNAP_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)

FILES = {
    "traffic": DATA_DIR / "合并表格，小时交通流量.csv",
    "temp": DATA_DIR / "合并表格，时间，气温，路温.csv",
    "holiday": DATA_DIR / "合并表格，holiday日级.csv",
    "weather": DATA_DIR / "合并表格，weather日级.csv",
    "construction": DATA_DIR / "合并表格，construction日级.csv",
    "events": DATA_DIR / "合并表格，special_events日级.csv",
}

print("ROOT      :", ROOT)
print("DATA_DIR  :", DATA_DIR)
for k, v in FILES.items():
    print(f"  {k:13s}: {'OK' if v.exists() else 'MISSING'}  {v.name}")

## 2. 数据加载与清洗

所有表分隔符 `;`，**第 2 行是中文说明**（读取时跳过），部分列用逗号小数。

In [ ]:
def read_semicolon(path: Path) -> pd.DataFrame:
    """读取分号分隔表，跳过第 2 行中文说明。"""
    return pd.read_csv(path, sep=";", skiprows=[1], dtype=str, keep_default_na=True)


def to_num(series: pd.Series) -> pd.Series:
    """逗号小数 -> 浮点。"""
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False).replace({"nan": np.nan, "": np.nan}),
        errors="coerce",
    )


# ---------- 2.1 主表：小时交通流量 ----------
traffic = read_semicolon(FILES["traffic"])
for c in ["bab_km", "longitude", "latitude", "kfz_h", "sv_h", "v_kfz"]:
    traffic[c] = to_num(traffic[c])

# 时间戳：datum=DD.MM.YYYY, t_start=HH:MM:SS
traffic["ts"] = pd.to_datetime(
    traffic["datum"] + " " + traffic["t_start"], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)
traffic = traffic.dropna(subset=["ts"]).copy()
traffic["date"] = traffic["ts"].dt.normalize()
traffic["hour"] = traffic["ts"].dt.hour
traffic["weekday"] = traffic["wochentag"].astype(int)          # 1-7
traffic["site_id"] = traffic["road"] + "_" + traffic["direction"] + "_" + traffic["site_name"]

# 异常处理：物理不可能值 -> NaN
traffic.loc[traffic["kfz_h"] < 0, "kfz_h"] = np.nan
traffic.loc[traffic["sv_h"] < 0, "sv_h"] = np.nan
# v_kfz 仅在 kfz_h>0 时有效
traffic.loc[(traffic["kfz_h"].isna()) | (traffic["kfz_h"] <= 0), "v_kfz"] = np.nan

print("主表行数:", len(traffic))
print("站点数  :", traffic["site_id"].nunique(), "->", sorted(traffic["site_id"].unique()))
print("时间范围:", traffic["ts"].min(), "~", traffic["ts"].max())
traffic[["site_id", "ts", "hour", "weekday", "tagestyp", "kfz_h", "sv_h", "v_kfz"]].head()

In [ ]:
# ---------- 2.2 温度表（分钟）-> 小时聚合 ----------
# t_start;lt;fbt   (lt=气温, fbt=路温)，dot 小数
temp_raw = read_semicolon(FILES["temp"])
temp_raw["lt"] = to_num(temp_raw["lt"])
temp_raw["fbt"] = to_num(temp_raw["fbt"])
temp_raw["ts"] = pd.to_datetime(temp_raw["t_start"], errors="coerce")
temp_raw = temp_raw.dropna(subset=["ts"]).copy()
temp_raw["date"] = temp_raw["ts"].dt.normalize()
temp_raw["hour"] = temp_raw["ts"].dt.hour

temp_hourly = (
    temp_raw.groupby(["date", "hour"])
    .agg(lt_mean=("lt", "mean"), fbt_mean=("fbt", "mean"), fbt_min=("fbt", "min"))
    .reset_index()
)

# 气候态 (month, hour) 备用：未来/缺测回填
temp_hourly["month"] = temp_hourly["date"].dt.month
temp_climo = (
    temp_hourly.groupby(["month", "hour"])
    .agg(lt_mean_c=("lt_mean", "mean"), fbt_mean_c=("fbt_mean", "mean"), fbt_min_c=("fbt_min", "mean"))
    .reset_index()
)
print("温度小时表:", temp_hourly.shape, "| 气候态:", temp_climo.shape)
temp_hourly.head()

In [ ]:
# ---------- 2.3 conditional 日级表 ----------
def load_daily(path: Path, num_cols, cat_cols):
    df = read_semicolon(path)
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    for c in num_cols:
        if c in df.columns:
            df[c] = to_num(df[c]).fillna(0)
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].fillna("").astype(str)
    keep = ["date"] + [c for c in (num_cols + cat_cols) if c in df.columns]
    return df[keep].drop_duplicates("date")


holiday = load_daily(
    FILES["holiday"],
    num_cols=[
        "is_school_holiday_DE_BY", "is_school_holiday_AT_SB", "is_school_holiday_AT_TI",
        "is_public_holiday_DE_BY", "is_public_holiday_AT_SB", "is_public_holiday_AT_TI",
        "school_holiday_count", "public_holiday_count",
        "is_holiday_start", "is_holiday_end", "in_traffic_window",
    ],
    cat_cols=["window_direction", "window_risk_level", "a8_direction", "a93_direction"],
)

weather = read_semicolon(FILES["weather"])
weather["date"] = pd.to_datetime(weather["date"], errors="coerce").dt.normalize()
for c in ["precip_mm", "snowfall_mm", "low_vis_hours", "t_min_c", "t_max_c", "has_ice_risk",
          "precip_mm_mean", "low_vis_hours_mean", "ice_risk_prob", "t_min_c_mean", "t_max_c_mean"]:
    if c in weather.columns:
        weather[c] = to_num(weather[c])
# observed 优先, 缺失回填 climatology
weather["w_precip"] = weather["precip_mm"].fillna(weather.get("precip_mm_mean"))
weather["w_snow"] = weather["snowfall_mm"].fillna(0)
weather["w_lowvis"] = weather["low_vis_hours"].fillna(weather.get("low_vis_hours_mean"))
weather["w_tmin"] = weather["t_min_c"].fillna(weather.get("t_min_c_mean"))
weather["w_tmax"] = weather["t_max_c"].fillna(weather.get("t_max_c_mean"))
weather["w_ice"] = weather["has_ice_risk"].fillna(weather.get("ice_risk_prob"))
weather["weather_source"] = weather["weather_source"].fillna("climatology").astype(str)
weather = weather[["date", "w_precip", "w_snow", "w_lowvis", "w_tmin", "w_tmax", "w_ice", "weather_source"]]

construction = load_daily(
    FILES["construction"],
    num_cols=[
        "has_a8_construction", "has_a93_construction", "a8_construction_count", "a93_construction_count",
        "has_2_plus_0", "two_plus_0_count", "max_closed_lanes", "sum_closed_lanes",
        "has_target_bbox_construction",
    ],
    cat_cols=[],
)

events = load_daily(
    FILES["events"],
    num_cols=[
        "has_special_event", "active_event_count", "max_impact_level", "impact_score",
        "affects_a8_ost", "affects_a93_sued",
        "has_munich_event", "has_salzburg_event", "has_rosenheim_event", "has_kufstein_event",
        "has_confirmed_event", "has_estimated_event",
    ],
    cat_cols=[],
)

print("holiday     :", holiday.shape)
print("weather     :", weather.shape)
print("construction:", construction.shape)
print("events      :", events.shape)
holiday.head(3)

## 3. 特征工程

- **日历特征**：周期 sin/cos 编码
- **历史画像特征**：从训练集按日历键聚合（out-of-fold，防泄漏）—— 主流量信号入模的核心方式
- **conditional 合并**：假期/天气/温度/施工/事件 left join

In [ ]:
# ---------- 3.1 日历特征 ----------
def add_calendar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    d = df["date"]
    df["month"] = d.dt.month
    df["doy"] = d.dt.dayofyear
    df["week_of_year"] = d.dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["weekday"] >= 6).astype(int)
    df["is_friday"] = (df["weekday"] == 5).astype(int)
    df["is_saturday"] = (df["weekday"] == 6).astype(int)
    df["is_sunday"] = (df["weekday"] == 7).astype(int)
    # 周期编码
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * (df["weekday"] - 1) / 7)
    df["dow_cos"] = np.cos(2 * np.pi * (df["weekday"] - 1) / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)
    # 季节
    season_map = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
                  6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}
    df["season"] = df["month"].map(season_map)
    return df


# ---------- 3.2 conditional + 温度 合并 ----------
def merge_conditional(df: pd.DataFrame) -> pd.DataFrame:
    df = df.merge(holiday, on="date", how="left")
    df = df.merge(weather, on="date", how="left")
    df = df.merge(construction, on="date", how="left")
    df = df.merge(events, on="date", how="left")
    df = df.merge(temp_hourly[["date", "hour", "lt_mean", "fbt_mean", "fbt_min"]],
                  on=["date", "hour"], how="left")
    # 温度缺失 -> (month, hour) 气候态
    df = df.merge(temp_climo, on=["month", "hour"], how="left")
    df["lt_mean"] = df["lt_mean"].fillna(df["lt_mean_c"])
    df["fbt_mean"] = df["fbt_mean"].fillna(df["fbt_mean_c"])
    df["fbt_min"] = df["fbt_min"].fillna(df["fbt_min_c"])
    df = df.drop(columns=["lt_mean_c", "fbt_mean_c", "fbt_min_c"])
    # 类别列空值补占位
    for c in ["weather_source", "window_direction", "window_risk_level", "a8_direction", "a93_direction"]:
        if c in df.columns:
            df[c] = df[c].fillna("none").replace("", "none").astype(str)
    return df


base = add_calendar(traffic)
base["lkw_ratio"] = np.where(base["kfz_h"] > 0, base["sv_h"] / base["kfz_h"], np.nan)
base = merge_conditional(base)
print("合并后特征表:", base.shape)
print("含 NaN 的特征列(前20):")
print(base.isna().sum().sort_values(ascending=False).head(20))
base.head(3)

In [ ]:
# ---------- 3.3 时序切分 ----------
train_mask = base["ts"] <= TRAIN_END
val_mask = base["ts"] >= VAL_START
train_df = base[train_mask].copy()
val_df = base[val_mask].copy()
print(f"训练集: {len(train_df):>8,} 行  ({train_df['ts'].min().date()} ~ {train_df['ts'].max().date()})")
print(f"验证集: {len(val_df):>8,} 行  ({val_df['ts'].min().date()} ~ {val_df['ts'].max().date()})")


# ---------- 3.4 历史画像特征（仅用训练集聚合，防泄漏）----------
def build_profiles(tr: pd.DataFrame) -> dict:
    tr = tr[tr["kfz_h"].notna()]
    return {
        "prof_kfz_shd": tr.groupby(["site_id", "hour", "weekday"])["kfz_h"].median().rename("prof_kfz_shd"),
        "prof_kfz_sht": tr.groupby(["site_id", "hour", "tagestyp"])["kfz_h"].median().rename("prof_kfz_sht"),
        "prof_kfz_shm": tr.groupby(["site_id", "hour", "month"])["kfz_h"].median().rename("prof_kfz_shm"),
        "prof_kfz_p90": tr.groupby(["site_id", "hour", "weekday"])["kfz_h"].quantile(PROFILE_KFZ_P90).rename("prof_kfz_p90"),
        "prof_lkw_shd": tr[tr["lkw_ratio"].notna()].groupby(["site_id", "hour", "weekday"])["lkw_ratio"].median().rename("prof_lkw_shd"),
        "prof_v_shd": tr[tr["v_kfz"].notna()].groupby(["site_id", "hour", "weekday"])["v_kfz"].median().rename("prof_v_shd"),
        "prof_v_p85": tr[tr["v_kfz"].notna()].groupby(["site_id", "hour"])["v_kfz"].quantile(PROFILE_V_P85).rename("prof_v_p85"),
    }


def apply_profiles(df: pd.DataFrame, profs: dict) -> pd.DataFrame:
    df = df.copy()
    keys = {
        "prof_kfz_shd": ["site_id", "hour", "weekday"],
        "prof_kfz_sht": ["site_id", "hour", "tagestyp"],
        "prof_kfz_shm": ["site_id", "hour", "month"],
        "prof_kfz_p90": ["site_id", "hour", "weekday"],
        "prof_lkw_shd": ["site_id", "hour", "weekday"],
        "prof_v_shd": ["site_id", "hour", "weekday"],
        "prof_v_p85": ["site_id", "hour"],
    }
    for name, k in keys.items():
        df = df.merge(profs[name], on=k, how="left")
    # 残余缺失用全局中位数兜底
    for name in keys:
        if df[name].isna().any():
            df[name] = df[name].fillna(df[name].median())
    return df


profiles = build_profiles(train_df)
train_df = apply_profiles(train_df, profiles)
val_df = apply_profiles(val_df, profiles)
print("画像特征已加入。示例列:", [c for c in train_df.columns if c.startswith("prof_")])

In [ ]:
# ---------- 3.5 特征清单 ----------
CALENDAR = [
    "hour", "weekday", "month", "doy", "week_of_year",
    "is_weekend", "is_friday", "is_saturday", "is_sunday",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
]
STATIC_NUM = ["bab_km", "longitude", "latitude"]
STATIC_CAT = ["site_id", "road", "direction", "site_name", "tagestyp", "season"]

HOLIDAY = [
    "is_school_holiday_DE_BY", "is_school_holiday_AT_SB", "is_school_holiday_AT_TI",
    "is_public_holiday_DE_BY", "is_public_holiday_AT_SB", "is_public_holiday_AT_TI",
    "school_holiday_count", "public_holiday_count",
    "is_holiday_start", "is_holiday_end", "in_traffic_window",
]
HOLIDAY_CAT = ["window_direction", "window_risk_level", "a8_direction", "a93_direction"]
WEATHER = ["w_precip", "w_snow", "w_lowvis", "w_tmin", "w_tmax", "w_ice"]
WEATHER_CAT = ["weather_source"]
TEMP = ["lt_mean", "fbt_mean", "fbt_min"]
CONSTRUCTION = [
    "has_a8_construction", "has_a93_construction", "a8_construction_count", "a93_construction_count",
    "has_2_plus_0", "two_plus_0_count", "max_closed_lanes", "sum_closed_lanes", "has_target_bbox_construction",
]
EVENTS = [
    "has_special_event", "active_event_count", "max_impact_level", "impact_score",
    "affects_a8_ost", "affects_a93_sued",
    "has_munich_event", "has_salzburg_event", "has_rosenheim_event", "has_kufstein_event",
    "has_confirmed_event", "has_estimated_event",
]

PROF_KFZ = ["prof_kfz_shd", "prof_kfz_sht", "prof_kfz_shm", "prof_kfz_p90"]
PROF_LKW = ["prof_lkw_shd"]
PROF_V = ["prof_v_shd", "prof_v_p85"]

COND = HOLIDAY + HOLIDAY_CAT + WEATHER + WEATHER_CAT + TEMP + CONSTRUCTION + EVENTS
CAT_FEATURES = STATIC_CAT + HOLIDAY_CAT + WEATHER_CAT

# 各目标的特征集合
FEATURES_KFZ = CALENDAR + STATIC_NUM + STATIC_CAT + PROF_KFZ + COND
FEATURES_LKW = CALENDAR + STATIC_NUM + STATIC_CAT + PROF_KFZ + PROF_LKW + COND
FEATURES_SPD = CALENDAR + STATIC_NUM + STATIC_CAT + PROF_KFZ + PROF_V + COND


def make_pool(df, features, target, weight=None):
    cats = [c for c in CAT_FEATURES if c in features]
    X = df[features].copy()
    for c in cats:
        X[c] = X[c].astype(str)
    y = df[target]
    w = df[weight] if weight else None
    return Pool(X, label=y, cat_features=cats, weight=w)


print("kfz 特征数:", len(FEATURES_KFZ), "| 类别特征:", [c for c in CAT_FEATURES if c in FEATURES_KFZ])

## 4. CatBoost 训练基础设施

- **tqdm 进度条**：自定义 callback，逐迭代刷新，显示 learn/val loss + 剩余时间(ETA)
- **快照续训**：`save_snapshot=True`，中断后重跑自动从断点继续
- **loss 记录**：`get_evals_result()` 供绘图

In [ ]:
class TqdmCallback:
    """CatBoost 训练进度条：显示 learn/val loss 与 ETA。"""

    def __init__(self, total: int, desc: str):
        self.bar = tqdm(total=total, desc=desc, leave=True, dynamic_ncols=True)
        self._keys = None

    def after_iteration(self, info) -> bool:
        self.bar.update(1)
        try:
            if self._keys is None:
                self._keys = list(info.metrics.get("learn", {}).keys())
            if self._keys:
                k = self._keys[0]
                post = {}
                learn = info.metrics.get("learn", {}).get(k)
                if learn:
                    post["learn"] = f"{learn[-1]:.3f}"
                val = info.metrics.get("validation", {}).get(k)
                if val:
                    post["val"] = f"{val[-1]:.3f}"
                if post:
                    self.bar.set_postfix(post)
        except Exception:
            pass
        return True

    def close(self):
        self.bar.close()


def train_catboost(train_pool, val_pool, loss_function, eval_metric, name, params=None):
    """训练单个 CatBoost 模型：tqdm 进度条 + 快照续训 + 返回 (model, evals_result)。"""
    p = dict(CB_PARAMS)
    if params:
        p.update(params)
    p.update(dict(
        loss_function=loss_function,
        eval_metric=eval_metric,
        train_dir=str(SNAP_DIR / name),
    ))
    if USE_SNAPSHOT:
        p.update(dict(
            save_snapshot=True,
            snapshot_file=str(SNAP_DIR / f"{name}.snapshot"),
            snapshot_interval=SNAPSHOT_INTERVAL_SEC,
        ))
    model = CatBoostRegressor(**p)
    cb = TqdmCallback(total=p["iterations"], desc=name)
    try:
        model.fit(train_pool, eval_set=val_pool, use_best_model=True,
                  callbacks=[cb], verbose=False)
    except Exception as e:
        cb.close()
        print(f"[{name}] callback 回退（{type(e).__name__}: {e}）")
        p_fb = dict(p)
        p_fb["save_snapshot"] = False
        model = CatBoostRegressor(**p_fb)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=FALLBACK_VERBOSE)
    else:
        cb.close()
    return model, model.get_evals_result()


print("训练工具就绪 ✔")

### 4.1 目标 1：总流量 `kfz_h`（P10 / P50 / P90 分位回归）

In [ ]:
# 去除目标缺失行
tr_kfz = train_df[train_df["kfz_h"].notna()]
va_kfz = val_df[val_df["kfz_h"].notna()]

pool_tr = make_pool(tr_kfz, FEATURES_KFZ, "kfz_h")
pool_va = make_pool(va_kfz, FEATURES_KFZ, "kfz_h")

kfz_models = {}
kfz_evals = {}
for q, alpha in QUANTILES.items():
    loss = f"Quantile:alpha={alpha}"
    model, ev = train_catboost(
        pool_tr, pool_va, loss_function=loss, eval_metric=loss,
        name=f"kfz_{q}", params=CB_KFZ_PARAMS,
    )
    kfz_models[q] = model
    kfz_evals[q] = ev
    model.save_model(str(MODEL_DIR / f"kfz_{q}.cbm"))
    print(f"  kfz_{q}: best_iter={model.get_best_iteration()}  -> models/kfz_{q}.cbm")

### 4.2 训练 / 验证 Loss 曲线

In [ ]:
for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    if _style in plt.style.available:
        plt.style.use(_style)
        break


def _first_metric(ev_part: dict):
    k = list(ev_part.keys())[0]
    return k, ev_part[k]


def plot_loss_curves(evals_dict: dict, suptitle: str, best_iters: dict | None = None):
    n = len(evals_dict)
    fig, axes = plt.subplots(1, n, figsize=(LOSS_FIGSIZE_PER_PANEL[0] * n, LOSS_FIGSIZE_PER_PANEL[1]), squeeze=False)
    axes = axes[0]
    palette = {"learn": "#2563eb", "val": "#dc2626"}
    for ax, (name, ev) in zip(axes, evals_dict.items()):
        mname, learn = _first_metric(ev["learn"])
        ax.plot(learn, label="train", color=palette["learn"], lw=1.8)
        if "validation" in ev:
            _, val = _first_metric(ev["validation"])
            ax.plot(val, label="validation", color=palette["val"], lw=1.8)
            bi = (best_iters or {}).get(name)
            if bi is not None and bi < len(val):
                ax.axvline(bi, color="#16a34a", ls="--", lw=1.2, alpha=0.8)
                ax.scatter([bi], [val[bi]], color="#16a34a", zorder=5,
                           label=f"best @ {bi}")
        ax.set_title(name, fontsize=12, fontweight="bold")
        ax.set_xlabel("iteration")
        ax.set_ylabel(mname)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
        ax.legend(frameon=True, fontsize=9)
    fig.suptitle(suptitle, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


plot_loss_curves(
    kfz_evals,
    "kfz_h · CatBoost 分位回归 Loss",
    best_iters={q: m.get_best_iteration() for q, m in kfz_models.items()},
)

### 4.3 目标 2：大车流量 `sv_h`（占比法）

预测 `lkw_ratio = sv_h / kfz_h`，再 `sv_h = lkw_ratio × kfz_h_p50`。

In [ ]:
tr_lkw = train_df[train_df["lkw_ratio"].notna()]
va_lkw = val_df[val_df["lkw_ratio"].notna()]

pool_tr_lkw = make_pool(tr_lkw, FEATURES_LKW, "lkw_ratio")
pool_va_lkw = make_pool(va_lkw, FEATURES_LKW, "lkw_ratio")

lkw_model, lkw_evals = train_catboost(
    pool_tr_lkw, pool_va_lkw,
    loss_function=CB_LKW_PARAMS["loss_function"],
    eval_metric=CB_LKW_PARAMS["eval_metric"],
    name="lkw_ratio", params=CB_LKW_PARAMS,
)
lkw_model.save_model(str(MODEL_DIR / "lkw_ratio.cbm"))
print("  -> models/lkw_ratio.cbm")

plot_loss_curves({"lkw_ratio (RMSE)": lkw_evals}, "sv_h 占比模型 Loss",
                 best_iters={"lkw_ratio (RMSE)": lkw_model.get_best_iteration()})

### 4.4 目标 3：平均车速 `v_kfz`（自由流基准 − 降速 两段式）

`speed_drop = prof_v_p85 − v_kfz`，预测降速后用 `v = base − drop` 还原，截断到 `[SPEED_CLIP_MIN, SPEED_CLIP_MAX]`（见 §1.1 超参数块）。

In [ ]:
# 降速目标
for d in (train_df, val_df):
    d["speed_drop"] = d["prof_v_p85"] - d["v_kfz"]

tr_spd = train_df[train_df["v_kfz"].notna()]
va_spd = val_df[val_df["v_kfz"].notna()]

pool_tr_spd = make_pool(tr_spd, FEATURES_SPD, "speed_drop")
pool_va_spd = make_pool(va_spd, FEATURES_SPD, "speed_drop")

spd_model, spd_evals = train_catboost(
    pool_tr_spd, pool_va_spd,
    loss_function=CB_SPD_PARAMS["loss_function"],
    eval_metric=CB_SPD_PARAMS["eval_metric"],
    name="speed_drop", params=CB_SPD_PARAMS,
)
spd_model.save_model(str(MODEL_DIR / "speed_drop.cbm"))
print("  -> models/speed_drop.cbm")

plot_loss_curves({"speed_drop (RMSE)": spd_evals}, "v_kfz 降速模型 Loss",
                 best_iters={"speed_drop (RMSE)": spd_model.get_best_iteration()})

## 5. 验证集评估（2025 hold-out）

MAE / RMSE / MAPE + 分位区间覆盖率 (PICP) + 峰值小时 Recall。

In [ ]:
def mae(y, p):
    return float(np.mean(np.abs(y - p)))

def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def mape(y, p, eps=MAPE_EPS):
    m = y > eps
    return float(np.mean(np.abs((y[m] - p[m]) / y[m])) * 100)


def predict_features(df, model, features):
    cats = [c for c in CAT_FEATURES if c in features]
    X = df[features].copy()
    for c in cats:
        X[c] = X[c].astype(str)
    return model.predict(X)


# ---- kfz_h ----
yk = va_kfz["kfz_h"].to_numpy()
pk = {q: predict_features(va_kfz, kfz_models[q], FEATURES_KFZ) for q in QUANTILES}
p50 = np.clip(pk["p50"], KFZ_CLIP_MIN, None)
picp = float(np.mean((yk >= pk["p10"]) & (yk <= pk["p90"])) * 100)
mpiw = float(np.mean(pk["p90"] - pk["p10"]))

# 峰值小时 recall
thr = np.quantile(yk, PEAK_QUANTILE)
true_peak = yk >= thr
pred_peak = p50 >= np.quantile(p50, PEAK_QUANTILE)
recall_peak = float(np.mean(pred_peak[true_peak])) if true_peak.any() else float("nan")

# ---- sv_h（占比 × kfz_p50）----
ys = va_lkw["sv_h"].to_numpy()
lkw_pred = np.clip(predict_features(va_lkw, lkw_model, FEATURES_LKW), LKW_RATIO_CLIP_MIN, None)
kfz_for_sv = np.clip(predict_features(va_lkw, kfz_models["p50"], FEATURES_KFZ), KFZ_CLIP_MIN, None)
sv_pred = lkw_pred * kfz_for_sv

# ---- v_kfz（base − drop）----
yv = va_spd["v_kfz"].to_numpy()
drop_pred = predict_features(va_spd, spd_model, FEATURES_SPD)
v_pred = np.clip(va_spd["prof_v_p85"].to_numpy() - drop_pred, SPEED_CLIP_MIN, SPEED_CLIP_MAX)

report = pd.DataFrame([
    {"target": "kfz_h (P50)", "MAE": mae(yk, p50), "RMSE": rmse(yk, p50), "MAPE%": mape(yk, p50)},
    {"target": "sv_h",        "MAE": mae(ys, sv_pred), "RMSE": rmse(ys, sv_pred), "MAPE%": mape(ys, sv_pred)},
    {"target": "v_kfz",       "MAE": mae(yv, v_pred), "RMSE": rmse(yv, v_pred), "MAPE%": mape(yv, v_pred)},
]).set_index("target").round(3)

print("=" * 56)
print("验证集 (2025) 评估")
print("=" * 56)
print(report)
print("-" * 56)
print(f"kfz P10–P90 区间覆盖率 PICP : {picp:.1f}%  (目标≈{PICP_TARGET_PCT:.0f}%)")
print(f"kfz P10–P90 平均区间宽度 MPIW: {mpiw:,.0f} 辆")
print(f"峰值小时 Recall (top{(1-PEAK_QUANTILE)*100:.0f}%)     : {recall_peak*100:.1f}%")

In [ ]:
# 可视化：某站点一周的预测 vs 实测（含 P10–P90 区间）
va_plot = va_kfz.copy()
va_plot["p10"], va_plot["p50"], va_plot["p90"] = pk["p10"], np.clip(pk["p50"], KFZ_CLIP_MIN, None), pk["p90"]

site = va_plot["site_id"].value_counts().index[0]
sub = va_plot[(va_plot["site_id"] == site) &
              (va_plot["ts"] >= PLOT_WEEK_START) & (va_plot["ts"] < PLOT_WEEK_END)].sort_values("ts")

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
ax.fill_between(sub["ts"], sub["p10"], sub["p90"], color="#93c5fd", alpha=0.45, label="P10–P90")
ax.plot(sub["ts"], sub["p50"], color="#1d4ed8", lw=1.8, label="预测 P50")
ax.plot(sub["ts"], sub["kfz_h"], color="#111827", lw=1.5, ls="--", label="实测 kfz_h")
ax.set_title(f"{site} · {PLOT_WEEK_START}~{PLOT_WEEK_END} 小时流量预测 vs 实测", fontsize=13, fontweight="bold")
ax.set_ylabel("kfz_h (辆/h)")
ax.legend(frameon=True)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


fi = pd.Series(kfz_models["p50"].get_feature_importance(), index=FEATURES_KFZ).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
fi.head(FI_TOP_K)[::-1].plot.barh(ax=ax, color="#0ea5e9")
ax.set_title(f"kfz_h P50 · 特征重要性 Top{FI_TOP_K}", fontsize=13, fontweight="bold")
ax.set_xlabel("importance")
fig.tight_layout()
plt.show()

## 6. 持久化与 2026–2029 推理

保存画像查找表与站点元信息；提供未来网格推理函数（`predict_grid`）。
> 实际批量生成 4 年全网格时，建议用全量 2023–2025 重训画像 + 模型后再调用。

In [ ]:
import pickle

# 站点元信息（构建未来网格用）
site_meta = (
    traffic.groupby("site_id")[["road", "direction", "site_name", "bab_km", "longitude", "latitude"]]
    .first().reset_index()
)
site_meta.to_parquet(PROC_DIR / "site_meta.parquet", index=False)

# 画像查找表
with open(PROC_DIR / "profiles.pkl", "wb") as f:
    pickle.dump({k: v.reset_index() for k, v in profiles.items()}, f)

print("已保存:")
print("  models/kfz_p10.cbm  kfz_p50.cbm  kfz_p90.cbm")
print("  models/lkw_ratio.cbm  speed_drop.cbm")
print("  processed/site_meta.parquet  processed/profiles.pkl")


def predict_grid(dates, hours=range(24)):
    """对给定日期 × 全部站点 × 小时 构建特征并预测三目标。
    dates: 可迭代的日期字符串/Timestamp（如 pd.date_range('2026-01-01','2029-12-31')）。
    返回带 kfz_h_p10/p50/p90, sv_h_pred, v_kfz_pred 的 DataFrame。
    """
    dates = pd.to_datetime(list(dates)).normalize()
    grid = pd.MultiIndex.from_product(
        [site_meta["site_id"], dates, list(hours)], names=["site_id", "date", "hour"]
    ).to_frame(index=False)
    grid = grid.merge(site_meta, on="site_id", how="left")
    grid["weekday"] = grid["date"].dt.weekday + 1
    grid["ts"] = grid["date"] + pd.to_timedelta(grid["hour"], unit="h")
    # tagestyp 用日历近似：周日=s，其余=w（更精细可由 holiday 表推导）
    grid["tagestyp"] = np.where(grid["weekday"] == 7, "s", "w")
    grid = add_calendar(grid)
    grid = merge_conditional(grid)        # 未来天气=climatology, 施工/事件未知=0
    grid = apply_profiles(grid, profiles)
    # 预测
    out = grid[["site_id", "road", "direction", "site_name", "date", "hour"]].copy()
    for q in QUANTILES:
        out[f"kfz_h_{q}"] = np.clip(predict_features(grid, kfz_models[q], FEATURES_KFZ), KFZ_CLIP_MIN, None)
    lkw = np.clip(predict_features(grid, lkw_model, FEATURES_LKW), LKW_RATIO_CLIP_MIN, None)
    out["sv_h_pred"] = lkw * out["kfz_h_p50"]
    drop = predict_features(grid, spd_model, FEATURES_SPD)
    out["v_kfz_pred"] = np.clip(grid["prof_v_p85"].to_numpy() - drop, SPEED_CLIP_MIN, SPEED_CLIP_MAX)
    out["interval_width"] = out["kfz_h_p90"] - out["kfz_h_p10"]
    return out


demo = predict_grid(pd.date_range(DEMO_FORECAST_DATE, DEMO_FORECAST_DATE))
print(f"\n{DEMO_FORECAST_DATE} 预测样例:")
demo.head(8)